<a href="https://colab.research.google.com/github/LourdesBranchi/camus-lv-segmentation/blob/main/notebooks/02_train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Entrenamiento en Google Colab — CAMUS Segmentación

Este notebook entrena los dos modelos y genera los resultados comparativos.

## Verificar GPU

In [42]:
import torch
print(f'GPU disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

GPU disponible: True
GPU: Tesla T4
Memoria: 15.6 GB


## Montar Drive y definir DRIVE_DIR

In [43]:
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/camus_checkpoints_v1'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints se guardarán en: {DRIVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints se guardarán en: /content/drive/MyDrive/camus_checkpoints_v1


## Clonar repositorio

In [46]:
%cd /content
!rm -rf /content/camus-lv-segmentation
!git clone https://github.com/LourdesBranchi/camus-lv-segmentation.git
%cd /content/camus-lv-segmentation
!pip install -r requirements.txt -q
print("Listo")

/content
Cloning into 'camus-lv-segmentation'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 55 (delta 21), reused 39 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 286.49 KiB | 8.95 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/camus-lv-segmentation
Listo


In [47]:
# Agregar src al path
import sys
sys.path.insert(0, '/content/camus-lv-segmentation/src')

## Configurar Kaggle desde secret y descargar dataset

In [48]:
from google.colab import userdata
import os, json

kaggle_token = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_CONFIG_DIR'] = '/root/.kaggle'
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    f.write(kaggle_token)
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d shoybhasan/camus-human-heart-data -q
!unzip -q camus-human-heart-data.zip -d /content/dataset_raw
!rm camus-human-heart-data.zip
print('Dataset descargado.')

Dataset URL: https://www.kaggle.com/datasets/shoybhasan/camus-human-heart-data
License(s): other
replace /content/dataset_raw/download? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
Dataset descargado.


In [50]:
from abc import ABCMeta
# Descomprimir dataset interno y definir DATA_ROOT
import zipfile, os

inner = '/content/dataset_raw/download'
if os.path.exists(inner):
    with zipfile.ZipFile(inner, 'r') as z:
      z.extractall('/content/datos_corazon')
    print('Dataset listo.')
else:
    print('Buscando archivo comprimido interno:')
    !find /content/dataset_raw -name '*.zip' -o -name 'download'

DATA_ROOT = '/content/datos_corazon/database_nifti'
print(f'DATA_ROOT = {DATA_ROOT}')
!ls {DATA_ROOT} | head -5

Dataset listo.
DATA_ROOT = /content/datos_corazon/database_nifti
patient0001
patient0002
patient0003
patient0004
patient0005


## EDA rápido

In [51]:
from dataset import prepare_dataset
import matplotlib.pyplot as plt
import numpy as np

loaders = prepare_dataset(
    data_root=DATA_ROOT,
    batch_size=8,
    num_workers=2,
)

imgs, masks = next(iter(loaders['train']))
print(f'Batch imágenes: {imgs.shape}  dtype={imgs.dtype}')
print(f'Batch máscaras: {masks.shape}  clases={masks.unique().tolist()}')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i in range(4):
    axes[0, i].imshow(imgs[i, 0].numpy(), cmap='gray')
    axes[0, i].set_title(f'Imagen {i+1}'); axes[0, i].axis('off')
    axes[1, i].imshow(masks[i].numpy(), cmap='jet', vmin=0, vmax=3)
    axes[1, i].set_title(f'Máscara {i+1}'); axes[1, i].axis('off')
plt.suptitle('Muestra del Dataset — Train Set', fontweight='bold')
plt.tight_layout()
plt.show()

Pacientes encontrados: 500
Imágenes totales:      2000

División de pacientes:
  Train: 350 pacientes
  Val:   75 pacientes
  Test:  75 pacientes

Pares imagen/máscara:
  Train: 1400
  Val:   300
  Test:  300


/content/camus-lv-segmentation/src/dataset.py:61: UserWarning: Argument(s) 'std_limit' are not valid for transform GaussNoise
  A.GaussNoise(std_limit=(3.16, 7.07), p=0.3),


Batch imágenes: torch.Size([8, 1, 256, 256])  dtype=torch.float32
Batch máscaras: torch.Size([8, 256, 256])  clases=[0, 1, 2, 3]


## Entrenar Modelo 1: U-Net + ResNet34

In [52]:
from train import train, DEFAULT_CONFIG

config_unet = {**DEFAULT_CONFIG,
    'model':      'unet',
    'encoder':    'resnet34',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  DATA_ROOT,
    'save_dir':   DRIVE_DIR,
    'device':     'cuda',
}

history_unet = train(config_unet)

Modelo:   unet + resnet34
Device:   cuda
Épocas:   50
Batch:    8
LR:       0.0001

Cargando dataset...
Pacientes encontrados: 500
Imágenes totales:      2000

División de pacientes:
  Train: 350 pacientes
  Val:   75 pacientes
  Test:  75 pacientes

Pares imagen/máscara:
  Train: 1400
  Val:   300
  Test:  300

Creando modelo...


/content/camus-lv-segmentation/src/dataset.py:61: UserWarning: Argument(s) 'std_limit' are not valid for transform GaussNoise
  A.GaussNoise(std_limit=(3.16, 7.07), p=0.3),


U-Net (resnet34, ImageNet) — 24.4M parámetros

Entrenando por 50 épocas...

Época [  1/50] (41s) lr=1.00e-04
  Train  loss=0.5966  dice=0.5754  [LV=0.753 MYO=0.619 LA=0.355]
  Val    loss=0.3316  dice=0.8388  [LV=0.896 MYO=0.808 LA=0.812]
  ✓ Mejor modelo guardado (val_dice=0.8388)

Época [  2/50] (41s) lr=1.00e-04
  Train  loss=0.2418  dice=0.8553  [LV=0.905 MYO=0.808 LA=0.853]
  Val    loss=0.1746  dice=0.8848  [LV=0.920 MYO=0.844 LA=0.890]
  ✓ Mejor modelo guardado (val_dice=0.8848)

Época [  3/50] (41s) lr=1.00e-04
  Train  loss=0.1596  dice=0.8770  [LV=0.918 MYO=0.834 LA=0.880]
  Val    loss=0.1368  dice=0.8874  [LV=0.920 MYO=0.849 LA=0.893]
  ✓ Mejor modelo guardado (val_dice=0.8874)

Época [  4/50] (42s) lr=1.00e-04
  Train  loss=0.1319  dice=0.8855  [LV=0.923 MYO=0.843 LA=0.891]
  Val    loss=0.1225  dice=0.8937  [LV=0.933 MYO=0.852 LA=0.896]
  ✓ Mejor modelo guardado (val_dice=0.8937)

Época [  5/50] (42s) lr=1.00e-04
  Train  loss=0.1133  dice=0.8961  [LV=0.930 MYO=0.857 LA=0

In [53]:
# Guardar history_unet como JSON en Drive
import json

unet_json_path = f'{DRIVE_DIR}/history_unet_resnet34.json'
with open(unet_json_path, 'w') as f:
    json.dump(history_unet, f)
print(f'history_unet guardado en: {unet_json_path}')

history_unet guardado en: /content/drive/MyDrive/camus_checkpoints_v1/history_unet_resnet34.json


## Entrenar Modelo 2: Attention U-Net + EfficientNet-B0

In [54]:
config_attn = {**DEFAULT_CONFIG,
    'model':      'attention-unet',
    'encoder':    'efficientnet-b0',
    'epochs':     50,
    'batch_size': 8,
    'lr':         1e-4,
    'patience':   10,
    'data_root':  DATA_ROOT,
    'save_dir':   DRIVE_DIR,
    'device':     'cuda',
}

history_attn = train(config_attn)

Modelo:   attention-unet + efficientnet-b0
Device:   cuda
Épocas:   50
Batch:    8
LR:       0.0001

Cargando dataset...
Pacientes encontrados: 500
Imágenes totales:      2000

División de pacientes:
  Train: 350 pacientes
  Val:   75 pacientes
  Test:  75 pacientes

Pares imagen/máscara:
  Train: 1400
  Val:   300
  Test:  300

Creando modelo...
Attention U-Net (efficientnet-b0, ImageNet) — 6.3M parámetros

Entrenando por 50 épocas...

Época [  1/50] (44s) lr=1.00e-04
  Train  loss=0.6375  dice=0.5524  [LV=0.641 MYO=0.591 LA=0.425]
  Val    loss=0.8530  dice=0.0051  [LV=0.000 MYO=0.000 LA=0.015]
  ✓ Mejor modelo guardado (val_dice=0.0051)

Época [  2/50] (44s) lr=1.00e-04
  Train  loss=0.2501  dice=0.8423  [LV=0.895 MYO=0.796 LA=0.835]
  Val    loss=0.9991  dice=0.0000  [LV=0.000 MYO=0.000 LA=0.000]

Época [  3/50] (44s) lr=1.00e-04
  Train  loss=0.1680  dice=0.8645  [LV=0.908 MYO=0.821 LA=0.865]
  Val    loss=1.0272  dice=0.0090  [LV=0.003 MYO=0.009 LA=0.015]
  ✓ Mejor modelo guardad

In [55]:
# Guardar history_attn como JSON en Drive
import json

attn_json_path = f'{DRIVE_DIR}/history_attn_efficientnet.json'
with open(attn_json_path, 'w') as f:
    json.dump(history_attn, f)
print(f'history_attn guardado en: {attn_json_path}')

history_attn guardado en: /content/drive/MyDrive/camus_checkpoints_v1/history_attn_efficientnet.json


## Graficar curvas de training de ambos modelos

In [57]:
from evaluate import plot_training_history

plot_training_history(history_unet, 'U-Net + ResNet34',                  f'{DRIVE_DIR}/unet_history.png')
plot_training_history(history_attn, 'Attention U-Net + EfficientNet-B0', f'{DRIVE_DIR}/attn_history.png')

Historial guardado en: /content/drive/MyDrive/camus_checkpoints_v1/unet_history.png
Historial guardado en: /content/drive/MyDrive/camus_checkpoints_v1/attn_history.png


## Evaluar ambos modelos en test set y guardar resultados

In [58]:
import torch, json
from models   import get_model
from losses   import CombinedLoss
from evaluate import evaluate, print_metrics_table

device  = torch.device('cuda')
loss_fn = CombinedLoss()
results = {}

# U-Net
ckpt1 = torch.load(f'{DRIVE_DIR}/best_unet_resnet34.pth', map_location=device)
m1 = get_model('unet', encoder_name='resnet34').to(device)
m1.load_state_dict(ckpt1['model_state'])
r1 = evaluate(m1, loaders['test'], loss_fn, device, 'U-Net')
print_metrics_table(r1, 'U-Net + ResNet34')
results['U-Net + ResNet34'] = r1

# Attention U-Net
ckpt2 = torch.load(f'{DRIVE_DIR}/best_attention-unet_efficientnet-b0.pth', map_location=device)
m2 = get_model('attention-unet', encoder_name='efficientnet-b0').to(device)
m2.load_state_dict(ckpt2['model_state'])
r2 = evaluate(m2, loaders['test'], loss_fn, device, 'Attention U-Net')
print_metrics_table(r2, 'Attention U-Net + EfficientNet-B0')
results['Attention U-Net + EfficientNet-B0'] = r2

# Guardar resultados numéricos en Drive
with open(f'{DRIVE_DIR}/test_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Resultados guardados en: {DRIVE_DIR}/test_results.json')

U-Net (resnet34, ImageNet) — 24.4M parámetros



  Resultados en Test Set — U-Net + ResNet34
  Métrica                        Valor
  -----------------------------------
  Loss                          0.0939
  -----------------------------------
  Mean Dice (cardíaco)          0.9090
  Dice LV                       0.9415
  Dice MYO                      0.8775
  Dice LA                       0.9079
  -----------------------------------
  Mean IoU (cardíaco)           0.8354
  IoU LV                        0.8898
  IoU MYO                       0.7823
  IoU LA                        0.8342

Attention U-Net (efficientnet-b0, ImageNet) — 6.3M parámetros



  Resultados en Test Set — Attention U-Net + EfficientNet-B0
  Métrica                        Valor
  -----------------------------------
  Loss                          0.0954
  -----------------------------------
  Mean Dice (cardíaco)          0.9045
  Dice LV                       0.9381
  Dice MYO                      0.8740
  Dice LA                       0.9014
  -----------------------------------
  Mean IoU (cardíaco)           0.8283
  IoU LV                        0.8839
  IoU MYO                       0.7771
  IoU LA                        0.8240

Resultados guardados en: /content/drive/MyDrive/camus_checkpoints_v1/test_results.json


## Visualizar predicciones para ambos modelos

In [59]:
from evaluate import visualize_predictions

visualize_predictions(m1, loaders['test'], device,
                      n_samples=6, save_path=f'{DRIVE_DIR}/predictions_unet.png',
                      model_name='U-Net + ResNet34')

visualize_predictions(m2, loaders['test'], device,
                      n_samples=6, save_path=f'{DRIVE_DIR}/predictions_attn.png',
                      model_name='Attention U-Net + EfficientNet-B0')

Visualización guardada en: /content/drive/MyDrive/camus_checkpoints_v1/predictions_unet.png
Visualización guardada en: /content/drive/MyDrive/camus_checkpoints_v1/predictions_attn.png


## Comparamos ambos modelos

In [60]:
from evaluate import compare_models

compare_models(results, save_path=f'{DRIVE_DIR}/comparativa_modelos.png')

print('\n' + '='*65)
print(f'{"Modelo":<40} {"LV":>6} {"MYO":>6} {"LA":>6} {"Mean":>6}')
print('='*65)
for name, m in results.items():
    print(f'{name:<40} {m["dice_lv"]:>6.3f} {m["dice_myo"]:>6.3f} {m["dice_la"]:>6.3f} {m["mean_dice_cardiac"]:>6.3f}')
print(f'{"Referencia (Leclerc 2019)":<40} {0.94:>6.3f} {0.85:>6.3f} {0.89:>6.3f} {0.89:>6.3f}')
print('='*65)

Comparativa guardada en: /content/drive/MyDrive/camus_checkpoints_v1/comparativa_modelos.png

Modelo                                       LV    MYO     LA   Mean
U-Net + ResNet34                          0.941  0.878  0.908  0.909
Attention U-Net + EfficientNet-B0         0.938  0.874  0.901  0.905
Referencia (Leclerc 2019)                 0.940  0.850  0.890  0.890


## Guardamos resultados en drive

In [61]:
import shutil, os

archivos_locales = [
    'predictions_unet.png',
    'predictions_attn.png',
    'comparativa_modelos.png',
    'unet_history.png',
    'attn_history.png',
]

for f in archivos_locales:
    if os.path.exists(f):
        shutil.copy(f, DRIVE_DIR)
        print(f'Copiado: {f} → {DRIVE_DIR}')
    else:
        print(f'No encontrado (ya guardado directo): {f}')

print(f'\n✓ Todos los resultados disponibles en: {DRIVE_DIR}')
!ls {DRIVE_DIR}

No encontrado (ya guardado directo): predictions_unet.png
No encontrado (ya guardado directo): predictions_attn.png
No encontrado (ya guardado directo): comparativa_modelos.png
No encontrado (ya guardado directo): unet_history.png
No encontrado (ya guardado directo): attn_history.png

✓ Todos los resultados disponibles en: /content/drive/MyDrive/camus_checkpoints_v1
attn_history.png
best_attention-unet_efficientnet-b0.pth
best_unet_resnet34.pth
comparativa_modelos.png
history_attn_efficientnet.json
history_unet_resnet34.json
last_attention-unet_efficientnet-b0.pth
predictions_attn.png
predictions_unet.png
test_results.json
unet_history.png
